# Przygotowanie sygnałów

Zdarzenie to czas, w którym pin jest w stanie aktywnym (0). Impulsy stanu aktywnego są łączone w jedno zdarzenie, jeśli przerwa między nimi jest krótsza niż `max_gap`, a zdarzenia krótsze niż `min_duration` są odrzucane. Parametry są osobne dla każdego pinu i zebrane w nazwane warianty (`VARIANTS`, obecny: `CURRENT_VARIANT`); warianty i role pinów są w `dms_analysis.signals`.

Arduino odczytuje piny co 250 ms, więc czasy początku i końca są dokładne do jednej próbki.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.overview import cleaning_figure
from dms_analysis.raw import load_session
from dms_analysis.signals import (
    CURRENT_VARIANT,
    DISPENSE_PIN,
    REFILL_PIN,
    VARIANTS,
    cleaned_changes,
    detect_events,
    events_table,
    summary,
)

DB_PATH = Path("../data/meas-signs-temps_2026-10-18_no-temps-problem/measurement-data-saver.db")
SESSION_ID = 9
PINS = [REFILL_PIN, DISPENSE_PIN]
FIGURE_WIDTH = 1200

pd.set_option("display.max_rows", 500)

In [ ]:
data = load_session(DB_PATH, SESSION_ID)
variant = VARIANTS[CURRENT_VARIANT]
signal_events = detect_events(data.pins, data.end, PINS, variant)
cleaned = cleaned_changes(signal_events.events, data.pins)

for pin in PINS:
    print(f"pin {pin}: {variant.rules(pin)}")
summary(signal_events)

## Odrzucone krótkie zdarzenia

In [ ]:
events_table(signal_events.too_short)

## Zdarzenia

In [ ]:
events_table(signal_events.events)

## Przed i po obróbce

Zaznaczenie zakresu przybliża wszystkie wykresy, dwuklik wraca do całości.

In [ ]:
figure = cleaning_figure(data, cleaned, f"{DB_PATH.parent.name}, sesja {SESSION_ID}", PINS, FIGURE_WIDTH)
display(figure)